# TP 3 : LayerNorm et MLP

L'attention mélange les positions. Ici c'est l'autre moitié d'un bloc : remettre chaque vecteur à l'échelle, le transformer tout seul, puis l'ajouter à ce qu'on avait. Rien de tout ça ne regarde les tokens voisins.


In [1]:
import torch
import torch.nn as nn

class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)   # échelle apprise, 1 au départ
        self.beta = torch.zeros(dim)   # décalage appris, 0 au départ

    def __call__(self, x):
        # dernier axe = les coordonnées d'un token, pas le batch ni le temps
        mu = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        xhat = (x - mu) / torch.sqrt(var + self.eps)
        return self.gamma * xhat + self.beta


## Étape 1 : Coder LayerNorm

On normalise sur le dernier axe, celui des coordonnées d'un token, pas sur le batch ni sur le temps. eps évite une division par zéro quand le vecteur est constant. gamma et beta sont une échelle et un décalage qu'on pourrait apprendre. Au départ ils valent 1 et 0, donc ils ne changent rien. unbiased=False divise la variance par N, comme nn.LayerNorm. Ici ce ne sont pas encore des paramètres du modèle, on vérifie seulement la formule.


In [2]:
torch.manual_seed(1337)
norm = LayerNorm1d(100)
x0 = torch.randn(32, 100)
y = norm(x0)

print("feature 0 sur le batch   ", float(y[:, 0].mean()), float(y[:, 0].std()))
print("features du premier token", float(y[0].mean()), float(y[0].std()))

ref = nn.LayerNorm(100)
ecart = (norm(x0) - ref(x0)).abs().max().item()
print("écart max avec nn.LayerNorm", ecart)


feature 0 sur le batch    0.14759676158428192 0.8847487568855286
features du premier token -2.38418573772492e-09 1.005033254623413
écart max avec nn.LayerNorm 4.76837158203125e-07


## Étape 2 : Sur quel axe

Avant même de lancer la cellule, c'est y[0], les 100 coordonnées du premier token, qui doit donner une moyenne proche de 0 et un écart-type proche de 1. La feature 0 lue sur les 32 exemples n'a pas à être normalisée. torch.std divise par N-1, donc on lit environ 1,005 et pas exactement 1. LayerNorm ne fait pas une BatchNorm : la fonction ne dépend pas des autres phrases du batch, ce qui arrange la génération, où le batch vaut 1.


In [3]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, facteur=4):
        super().__init__()
        cachee = facteur * n_embd  # 4C par défaut : plus de place, puis on revient à C
        self.net = nn.Sequential(
            nn.Linear(n_embd, cachee),
            nn.ReLU(),  # met à zéro la moitié négative
            nn.Linear(cachee, n_embd),  # retour à C, pour pouvoir ajouter à x
        )

    def forward(self, x):
        return self.net(x)

ff = FeedForward(100)
lot = torch.randn(32, 100)
pre = ff.net[0](lot)
print("sortie", tuple(ff(lot).shape))
print(f"ReLU allumées : {(pre > 0).float().mean().item():.0%}")

# modifier le token 0 ne doit changer que la position 0
torch.manual_seed(0)
bloc = torch.randn(1, 4, 32)
f = FeedForward(32)
autre = bloc.clone()
autre[0, 0] += 5
delta = (f(bloc) - f(autre)).abs().sum(-1)
print("changement par position", delta.round(decimals=4).tolist())


sortie (32, 100)
ReLU allumées : 50%
changement par position [[21.778400421142578, 0.0, 0.0, 0.0]]


## Étape 3 : Le feed-forward

À l'initialisation, environ la moitié des activations de la couche large sont positives : ReLU éteint le reste. Passer par 4C laisse quand même de l'ordre de 2C unités allumées. On revient à C pour pouvoir écrire x + F(x), et pour que la couche suivante sache quelle largeur lire. La sortie a donc la même forme que l'entrée.

Le vecteur de changements est nul partout sauf à la position qu'on a modifiée. Le MLP ne mélange pas les tokens. Ça, c'est le travail de l'attention.


In [4]:
torch.manual_seed(1337)
B, T, n_embd = 4, 8, 32
x = torch.randn(B, T, n_embd)
ln = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)
delta = ffwd(ln(x))  # le déplacement, même forme que x
sortie = x + delta     # résidu : on garde x et on ajoute seulement la correction

def corr(u, v):
    u, v = u.flatten(), v.flatten()
    u, v = u - u.mean(), v - v.mean()
    return (torch.dot(u, v) / (u.norm() * v.norm())).item()

print(tuple(x.shape), tuple(delta.shape), tuple(sortie.shape))
print(f"corr(x, x+delta) {corr(x, sortie):.3f}")
print(f"corr(x, delta)   {corr(x, delta):.3f}")

class FeedForwardGELU(FeedForward):
    def __init__(self, n_embd, facteur=4):
        super().__init__(n_embd, facteur)
        cachee = facteur * n_embd
        self.net = nn.Sequential(
            nn.Linear(n_embd, cachee),
            nn.GELU(),  # comme ReLU, mais les valeurs négatives ne sont pas coupées net
            nn.Linear(cachee, n_embd),
        )

print("GELU", tuple(FeedForwardGELU(n_embd)(x).shape))

def compter(facteur, n=32):
    return sum(p.numel() for p in nn.Linear(n, facteur * n).parameters()) \
         + sum(p.numel() for p in nn.Linear(facteur * n, n).parameters())

for facteur in (2, 4, 8):
    print(f"C -> {facteur}C -> C : {compter(facteur)} paramètres")


(4, 8, 32) (4, 8, 32) (4, 8, 32)
corr(x, x+delta) 0.974
corr(x, delta)   0.025
GELU (4, 8, 32)
C -> 2C -> C : 4192 paramètres
C -> 4C -> C : 8352 paramètres
C -> 8C -> C : 16672 paramètres


## Étape 4 : Résidu, GELU, largeur

LayerNorm garde la dimension : (B, T, C) reste (B, T, C). Le MLP non plus ne mélange pas les positions, on l'a vu à l'étape 3. Enlever l'addition ne change pas la forme, F(x) est déjà en dimension C. Ce qui change, c'est le rôle : avec le résidu, la sortie reste une correction de l'entrée, elle est très corrélée à x. Sans lui, le MLP devrait reconstruire tout le vecteur. Chaque bloc n'apprend donc qu'un déplacement, et le gradient a un chemin direct.

GELU remplace la coupure nette de ReLU par une rampe plus lisse. La forme ne change pas. Un Linear(a, b) compte a*b poids plus b biais. C -> 4C -> C fait 8C² + 5C. Passer à 8C double presque ce coût, et le MLP est déjà le gros d'un bloc. Le facteur 4 est le choix habituel, pas une obligation.
